# MMMU Baseline on Colab A100 (VS Code에서 실행)

VS Code에서 이 파일을 열고 → 오른쪽 위 커널 선택 → **Colab 런타임** → **A100** 선택.

파이프라인: MMMU-Pro 공식 CoT 프롬프트 + 혼합 파서(`Answer:` 줄 → Llama-3.1-8B judge → 오답) + Qwen 공식 sampling recipe(seed 42, max_new_tokens 32768).
자세한 내용은 `README_mmmu_eval.md` 참고.

**미리 할 것**
1. https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct 에서 라이선스 동의 (judge 모델이 gated)
2. Colab 왼쪽 🔑(Secrets)에 `HF_TOKEN` 등록

## 1. GPU 확인

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemExit("GPU 런타임이 아닙니다. Colab 런타임에서 A100을 고르세요.")

name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"\nGPU: {name} | compute capability: {cap} | VRAM: {vram:.1f} GB")

if cap[0] < 8:
    print("⚠️  bf16 미지원 GPU(T4 등) — 과제 스펙(bf16) 위반이 되므로 A100/L4로 바꾸세요")

## 2. 코드 올리기 (Drive의 zip 한 개)

Mac에서 만든 `mmmu_project.zip`을 브라우저로 Google Drive **`내 드라이브` 최상위**에 올려 두고 실행합니다.
zip 안에는 `mmmu_eval/`, `scripts/`, `requirements.txt`, `github_predictions.jsonl`(8번 비교용)이 들어 있습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP     = '/content/drive/MyDrive/mmmu_project.zip'   # 업로드한 zip 위치
PROJECT = '/content/drive/MyDrive/mmmu_project'       # 결과와 lock 파일을 둘 Drive 폴더
WORK    = '/content/mmmu'                             # 런타임 로컬 디스크 (코드 실행 위치)

!mkdir -p {PROJECT} {WORK}
!unzip -o -q {ZIP} -d {WORK}
!ls -R {WORK} | head -30

## 3. 설치

- vllm 설치 시 torch 버전이 바뀌므로, 설치 후 **커널을 한 번 재시작**하고 4번 셀부터 이어서 실행하세요.
- 재현성 근거로 보고서 §1에 붙일 lock 파일을 Drive에 남깁니다.

In [ ]:
!pip install -q -r /content/mmmu/requirements.txt
!pip freeze > /content/drive/MyDrive/mmmu_project/requirements.lock.txt
print("설치 완료 → 커널 재시작 후 4번 셀부터 실행하세요")

## 4. 경로와 토큰

- `OUT_ROOT`(결과)는 **Drive**에 둡니다. 세션이 끊겨도 같은 셀을 다시 실행하면 끝난 문항은 건너뜁니다(resume).
- `DATA_ROOT`(MMMU HF 캐시)는 런타임 로컬 디스크에 둡니다. `load_dataset(과목, split="validation")`이 test split까지 받아 3GB 이상이라 세션마다 새로 받는 편이 빠릅니다.
- 모델 가중치(Qwen 약 9GB, judge 약 16GB)는 Drive에서 읽으면 느려서 매 세션 HF에서 받습니다.

In [ ]:
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # Colab Secrets에 HF_TOKEN 등록. 토큰을 코드에 직접 쓰지 말 것
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'

WORK      = '/content/mmmu'
DATA_ROOT = '/content/hf_mmmu'   # 런타임 로컬 디스크 (MMMU는 과목마다 test split까지 받아 3GB 이상 → Drive에 두면 느림)
OUT_ROOT  = '/content/drive/MyDrive/mmmu_project/results'
MODEL     = 'Qwen/Qwen3-VL-4B-Instruct'
REVISION  = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_ROOT, exist_ok=True)
%cd {WORK}

## 5. 스모크 테스트 — 본 실행 전 반드시

과목당 2문항(60문항)으로 서버 기동 → 추론 → judge → 채점까지 한 바퀴 도는지 확인하고, **속도와 VRAM을 실측**합니다.
**이 결과는 baseline 수치로 쓸 수 없습니다.**

In [ ]:
!pip uninstall -y torchaudio
!python -c "import torch, torchvision, vllm; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| torchvision', torchvision.__version__, '| vllm', vllm.__version__)"

In [ ]:
!bash scripts/run_mmmu_eval.sh \
  --model_path {MODEL} --model_revision {REVISION} \
  --data_root {DATA_ROOT} \
  --output_dir {OUT_ROOT}/smoke \
  --limit 2

## 6. 본 실행 — Run B (MMMU-Pro 공식 CoT, 900문항)

A100 추정 약 40분~1.5시간(스모크 결과로 다시 추정할 것). 끊기면 이 셀을 그대로 다시 실행하세요.

In [ ]:
!bash scripts/run_mmmu_eval.sh \
  --model_path {MODEL} --model_revision {REVISION} \
  --data_root {DATA_ROOT} \
  --output_dir {OUT_ROOT}/baseline_mmmu_pro_cot

## 7. 결과 확인 — 보고서 §1, §5 재료

In [ ]:
import json

def show(out_dir):
    s = json.load(open(f'{out_dir}/summary.json'))
    print(f"overall (macro)   : {s['overall_accuracy']*100:.2f}%   <- 보고용 대표값")
    print(f"overall (micro)   : {s['micro_accuracy']*100:.2f}%   <- 검산용 (macro와 같아야 정상)")
    print(f"rule-only (macro) : {s['rule_only_macro_accuracy']*100:.2f}%")
    print(f"parser methods    : {s['parser_method_counts']}")
    print(f"finish reasons    : {s['finish_reason_counts']}   truncation_rate={s['truncation_rate']:.3f}")
    print(f"infer wall        : {s['timing']['infer_wall_sec_total']/60:.1f} min | judge wall: {s['timing']['judge_wall_sec_total']/60:.1f} min")
    print(f"peak GPU mem (MiB): {s['peak_gpu_mem_mib']}\n")
    for cat, v in s['category_results'].items():
        print(f"{cat:<32}{v['samples']:>5}{v['accuracy']*100:>8.2f}%")
    print()
    print(f"{'subject':<40}{'n':>4}{'acc':>8}{'rule-only':>11}{'trunc':>7}")
    for r in s['subject_results']:
        print(f"{r['subject']:<40}{r['samples']:>4}{r['accuracy']*100:>7.1f}%{r['accuracy_rule_only']*100:>10.1f}%{r['truncated']:>7}")

show(f'{OUT_ROOT}/baseline_mmmu_pro_cot')

## 8. GitHub baseline과 비교

- GitHub 원래 수치(62.44%)는 seed(3407), max_new_tokens(4096), 파서가 모두 다릅니다.
- 아래 셀은 GitHub 응답을 **우리 혼합 파서(rule 단계)**로 다시 채점해서 파서 차이를 분리합니다. (`github_predictions.jsonl`은 zip에 들어 있습니다.)
- 900문항 기준 95% CI는 약 ±3.1%p이므로 그보다 작은 차이는 의미가 없습니다.

In [ ]:
!python scripts/rescore_github.py \
  --predictions /content/mmmu/github_predictions.jsonl \
  --data-root {DATA_ROOT} \
  --output {OUT_ROOT}/github_rescored.json | head -8

### (선택) Run A — GitHub 프롬프트를 같은 조건으로

프롬프트만 다르고 나머지(seed 42, 32768, 파서, judge)는 같게 해서 공정하게 비교합니다.

In [ ]:
!bash scripts/run_mmmu_eval.sh \
  --model_path {MODEL} --model_revision {REVISION} \
  --data_root {DATA_ROOT} \
  --output_dir {OUT_ROOT}/runA_github_cot \
  --prompt github_cot

show(f'{OUT_ROOT}/runA_github_cot')

## 9. Run A와 Run B 문항 단위 비교 (baseline 결정용)

GPU 없이 돌아갑니다. 새 세션이면 **2번 셀(zip 풀기)과 4번 셀(경로 설정)만** 먼저 실행하세요.
- McNemar 정확 검정: 한쪽만 맞힌 문항 수로 두 실행의 차이가 우연인지 판단 (p < 0.05면 유의)
- 잘림 교차표: 32768 토큰 잘림(반복 루프)이 두 실행에서 어떻게 겹치는지
- 과목별 차이

In [ ]:
!python scripts/compare_runs.py \
  --run-a {OUT_ROOT}/runA_github_cot \
  --run-b {OUT_ROOT}/baseline_mmmu_pro_cot \
  --output {OUT_ROOT}/compare_A_vs_B.json